In [ ]:
import sympy as sp
from SymEigen import *
import diff_friction
import sys
sys.path.append('../')
from project_dir import *

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction('__host__ __device__')
Gen.DisableLatexComment()

N = 4
Xn = Eigen.Matrix('Xn', 3 * N, 1)
X = Eigen.Matrix('X', 3 * N, 1)
dX = X - Xn
dfdtau = Eigen.Matrix('dfdtau', 2, 1)

Cl = Gen.Closure(Xn)

P = Xn[0:3, :]
T0 = Xn[3:6, :]
T1 = Xn[6:9, :]
T2 = Xn[9:12, :]

e1 = T1 - T0
e2 = T2 - T0
e3 = P - T0

D = e1.dot(e1) * e2.dot(e2) - (e1.dot(e2))**2
D1 = e1.dot(e3) * e2.dot(e2) - e1.dot(e2) * e2.dot(e3)
D2 = e1.dot(e1) * e2.dot(e3) - e1.dot(e3) * e1.dot(e2)

beta_1 = D1 / D
beta_2 = D2 / D

beta_0 = 1 - beta_1 - beta_2

e1_hat = Eigen.Normalize(e1)
e2_hat = Eigen.Normalize(e1.cross((e1.cross(e2))))

T = sp.Matrix.zeros(2, 3 * N)
T.shape

In [ ]:

T[0, 0:3] = e1_hat.T
T[1, 0:3] = e2_hat.T

T[0, 3:6] = -beta_0 * e1_hat.T
T[1, 3:6] = -beta_0 * e2_hat.T

T[0, 6:9] = -beta_1 * e1_hat.T
T[1, 6:9] = -beta_1 * e2_hat.T

T[0, 9:12] = -beta_2 * e1_hat.T
T[1, 9:12] = -beta_2 * e2_hat.T

In [ ]:
Cl = Gen.Closure(Xn)
content = ''
content += Cl('TR',T) + '\n'

In [ ]:
dTdXn = diff_friction.compute_dTdXn(T, Xn)
dTdXn.shape

In [ ]:
A, B = diff_friction.compute_AB(dTdXn, dX, dfdtau)
A.shape, B.shape

In [ ]:
Cl = Gen.Closure(Xn, X)
content += Cl('An', A) + '\n'

In [ ]:
Cl = Gen.Closure(Xn, X, dfdtau)
content += Cl('Bn', B) + '\n'

In [ ]:
filename = 'ipc_diff_friction_PT'
cuda_backend_source_dir = backend_source_dir('cuda')
f = open(f'{cuda_backend_source_dir}/contact_system/contact_models/sym/{filename}.inl', 'w')
f.write(content)
f.close()